## Lab 3: Multi-Source Retail Sales Data Integration and Analysis

## 1. Install Required Packages

In [1]:
install.packages(c(
  "tidyverse",
  "jsonlite",
  "readxl",
  "DBI",
  "RSQLite"
))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## 2. Load Required Libraries

In [2]:
library(tidyverse)
library(jsonlite)
library(readxl)
library(DBI)
library(RSQLite)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘jsonlite’


The following object is masked from ‘package:purrr’:

    flatten




## 3. Download UCI Online Retail Dataset

In [3]:
url <- "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

download.file(url, "online_retail.zip", mode = "wb")

unzip("online_retail.zip")

list.files()

[1] "Online Retail.xlsx" "online_retail.zip"  "sample_data"

## 4. Read the Dataset

In [4]:
retail <- read_excel("Online Retail.xlsx")

head(retail)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


## 5. Inspect the Dataset

In [5]:
dim(retail)

str(retail)

summary(retail)

colnames(retail)

[1] 541909      8

tibble [541,909 × 8] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr [1:541909] "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 $ UnitPrice  : num [1:541909] 2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Country    : chr [1:541909] "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


     InvoiceNo          StockCode         Description        Quantity         
 Length   :541909   Length   :541909   Length   :541909   Min.   :-80995.000  
 N.unique : 25900   N.unique :  4070   N.unique :  4211   1st Qu.:     1.000  
 N.blank  :     0   N.blank  :     0   N.blank  :     0   Median :     3.000  
 Min.nchar:     6   Min.nchar:     1   Min.nchar:     1   Mean   :     9.552  
 Max.nchar:     7   Max.nchar:    12   Max.nchar:    35   3rd Qu.:    10.000  
                                       NAs      :  1454   Max.   : 80995.000  
                                                                              
  InvoiceDate                    UnitPrice            CustomerID    
 Min.   :2010-12-01 08:26:00   Min.   :-11062.060   Min.   :12346   
 1st Qu.:2011-03-28 11:34:00   1st Qu.:     1.250   1st Qu.:13953   
 Median :2011-07-19 17:17:00   Median :     2.080   Median :15152   
 Mean   :2011-07-04 13:34:57   Mean   :     4.611   Mean   :15288   
 3rd Qu.:2011-10-19 11:

[1] "InvoiceNo"   "StockCode"   "Description" "Quantity"    "InvoiceDate"
[6] "UnitPrice"   "CustomerID"  "Country"

## 6. Rename Columns

In [6]:
retail <- retail %>%
  rename(
    InvoiceNo = InvoiceNo,
    StockCode = StockCode,
    Description = Description,
    Quantity = Quantity,
    InvoiceDate = InvoiceDate,
    UnitPrice = UnitPrice,
    CustomerID = CustomerID,
    Country = Country
  )

## 7. Create Transactions, Products and Customers Datasets

In [7]:
transactions <- retail %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate
  )

products <- retail %>%
  select(
    StockCode,
    Description,
    UnitPrice
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

customers <- retail %>%
  select(
    CustomerID,
    Country
  ) %>%
  distinct(CustomerID, .keep_all = TRUE)

## 8. Check Missing Values

In [8]:
colSums(is.na(transactions))
colSums(is.na(products))
colSums(is.na(customers))

InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0

StockCode Description   UnitPrice 
          0         176           0

CustomerID    Country 
         1          0

## 9. Clean Transactions Data

In [9]:
transactions <- transactions %>%
  filter(
    !is.na(CustomerID),
    Quantity > 0
  ) %>%
  distinct()

## 10. Clean Products Data

In [10]:
products <- products %>%
  filter(
    !is.na(StockCode),
    !is.na(UnitPrice),
    UnitPrice > 0
  ) %>%
  distinct()


## 11. Clean Customers Data

In [11]:
customers <- customers %>%
  filter(
    !is.na(CustomerID),
    !is.na(Country)
  ) %>%
  distinct()

## 12. Calculate Revenue

In [12]:
sales <- transactions %>%
  left_join(
    products,
    by = "StockCode"
  ) %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

head(sales)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30


## 13. Integrate Customer Information

In [13]:
final_data <- sales %>%
  left_join(
    customers,
    by = "CustomerID"
  )

head(final_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue,Country
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<chr>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30,United Kingdom
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34,United Kingdom
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00,United Kingdom
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34,United Kingdom
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34,United Kingdom
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30,United Kingdom


## 14. Verify Final Dataset Dimensions

In [14]:
dim(final_data)

[1] 392708      9

## 15. Identify Unmatched Records

In [15]:
sum(is.na(final_data$Description))

sum(is.na(final_data$UnitPrice))

sum(is.na(final_data$Country))

[1] 4831

[1] 4831

[1] 0

## 16. Calculate Total Sales Revenue

In [16]:
total_revenue <- final_data %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE)
  )

total_revenue

Total_Revenue
<dbl>
10752840



## 17. Find Top 5 Products by Revenue

In [17]:
top_products <- final_data %>%
  group_by(StockCode, Description) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top_products

StockCode,Description,Revenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
47566,PARTY BUNTING,142437.56
22423,REGENCY CAKESTAND 3 TIER,135604.80
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64


## 18. Find Top 5 Countries by Revenue

In [18]:
top_countries <- final_data %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top_countries

Country,Revenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


## 19. Find Top 5 Customers by Purchase Value

In [19]:
top_customers <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Purchase_Value)) %>%
  slice_head(n = 5)

top_customers

CustomerID,Total_Purchase_Value
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


## 20. Calculate Customer Purchase Value Thresholds

In [20]:
customer_values <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  )

quantile(
  customer_values$Total_Purchase_Value,
  probs = c(0.25, 0.50, 0.75),
  na.rm = TRUE
)

25%      50%      75% 
 361.005  802.600 2000.640

## 21. Classify Customers Using case_when()

In [21]:
q <- quantile(
  customer_values$Total_Purchase_Value,
  probs = c(0.25, 0.50, 0.75),
  na.rm = TRUE
)

customer_values <- customer_values %>%
  mutate(
    Customer_Category = case_when(
      Total_Purchase_Value <= q[1] ~ "Low Value",
      Total_Purchase_Value <= q[2] ~ "Medium Value",
      Total_Purchase_Value <= q[3] ~ "High Value",
      TRUE ~ "Premium"
    )
  )

head(customer_values)

CustomerID,Total_Purchase_Value,Customer_Category
<dbl>,<dbl>,<chr>
12346,77183.60,Premium
12347,5438.44,Premium
12348,1790.16,High Value
12349,1926.96,High Value
12350,408.04,Medium Value
12352,1616.57,High Value


## 22. Add Customer Classification

In [22]:
final_data <- final_data %>%
  left_join(
    customer_values %>%
      select(CustomerID, Customer_Category),
    by = "CustomerID"
  )

head(final_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue,Country,Customer_Category
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<chr>,<chr>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30,United Kingdom,Premium
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34,United Kingdom,Premium
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00,United Kingdom,Premium
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34,United Kingdom,Premium
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34,United Kingdom,Premium
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30,United Kingdom,Premium


## 23. Identify High-Performing Market

In [23]:
market_performance <- final_data %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue))

head(market_performance, 1)

Country,Revenue
<chr>,<dbl>
United Kingdom,8861857


## 24. Identify Underperforming Market

In [24]:
tail(market_performance, 1)

Country,Revenue
<chr>,<dbl>
Saudi Arabia,181.44


## 25. Create SQLite Database

In [25]:
con <- dbConnect(
  SQLite(),
  "retail_sales.db"
)

## 26. Export Data to SQLite

In [26]:
dbWriteTable(
  con,
  "retail_sales",
  final_data,
  overwrite = TRUE
)

## 27. Verify SQLite Table

In [27]:
dbListTables(con)

[1] "retail_sales"

## 28. SQL Query 1: Top 5 Customers

In [28]:
query1 <- dbGetQuery(
  con,
  "
  SELECT
    CustomerID,
    SUM(Revenue) AS Total_Revenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY Total_Revenue DESC
  LIMIT 5
  "
)

query1

CustomerID,Total_Revenue
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


## 29. SQL Query 2: Total Revenue by Country

In [30]:
query2 <- dbGetQuery(
  con,
  "
  SELECT
    Country,
    SUM(Revenue) AS Total_Revenue
  FROM retail_sales
  GROUP BY Country
  ORDER BY Total_Revenue DESC
  "
)

head(query2, 10)

,Country,Total_Revenue
,<chr>,<dbl>
1,United Kingdom,8861857.13
2,Netherlands,363884.48
3,EIRE,331660.17
4,Germany,263818.97
5,France,226975.60
6,Australia,173918.61
7,Spain,67426.09
8,Switzerland,66619.97
9,Japan,48600.22


## 30. Close SQLite Database

In [31]:
dbDisconnect(con)

## 31. Export Processed Datasets

In [33]:
install.packages("writexl")
library(writexl)

write_csv(
  transactions,
  "transactions.csv"
)

write_json(
  products,
  "products.json",
  pretty = TRUE,
  auto_unbox = TRUE
)

write_xlsx(
  customers,
  "customers.xlsx"
)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## 32. Business Insights

In [34]:
cat("Total Revenue:", total_revenue$Total_Revenue, "\n\n")

cat("Top Product:\n")
print(top_products[1, ])

cat("\nTop Country:\n")
print(top_countries[1, ])

cat("\nTop Customer:\n")
print(top_customers[1, ])

cat("\nHigh Performing Market:\n")
print(head(market_performance, 1))

cat("\nUnderperforming Market:\n")
print(tail(market_performance, 1))

Total Revenue: 10752840 

Top Product:
# A tibble: 1 × 3
  StockCode Description                 Revenue
  <chr>     <chr>                         <dbl>
1 23843     PAPER CRAFT , LITTLE BIRDIE 168470.

Top Country:
# A tibble: 1 × 2
  Country         Revenue
  <chr>             <dbl>
1 United Kingdom 8861857.

Top Customer:
# A tibble: 1 × 2
  CustomerID Total_Purchase_Value
       <dbl>                <dbl>
1      18102              408760.

High Performing Market:
# A tibble: 1 × 2
  Country         Revenue
  <chr>             <dbl>
1 United Kingdom 8861857.

Underperforming Market:
# A tibble: 1 × 2
  Country      Revenue
  <chr>          <dbl>
1 Saudi Arabia    181.
